# Logistic Regression — Base vs Google Trends

**Part 1** trains and evaluates a logistic regression on price + engineered features.  
**Part 2** adds 5 Google Trends features and evaluates independently.  
**Part 3** compares both models head-to-head.

In [30]:
import pathlib
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    f1_score, log_loss, precision_recall_curve, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [31]:
ROOT            = pathlib.Path("../..")
DATA_DIR        = ROOT / "data"
ARTIFACTS_DIR   = pathlib.Path("artifacts")
PREDICTIONS_DIR = pathlib.Path("predictions")

NUMERIC_FEATURES = [
    "price_at_snapshot",
    "price_deviation_from_half",
    "days_before_close",
    "pct_lifetime_elapsed",
    "duration_days",
    "log_volume",
    "price_mean_7d",   "price_volatility_7d",  "price_min_7d",  "price_max_7d",
    "price_change_7d", "price_range_7d",        "price_trend_7d",
    "price_mean_14d",  "price_volatility_14d", "price_min_14d", "price_max_14d",
    "price_change_14d","price_range_14d",       "price_trend_14d",
]
TRENDS_FEATURES      = ["trend_value", "trend_ma4", "trend_change_4w", "trend_spike", "has_trend_data"]
CATEGORICAL_FEATURES = ["category"]
TARGET               = "outcome"

FEATURES_BASE   = NUMERIC_FEATURES + CATEGORICAL_FEATURES
FEATURES_TRENDS = NUMERIC_FEATURES + TRENDS_FEATURES + CATEGORICAL_FEATURES

---
## Load Data

The trends-enriched dataset contains all base features plus the 5 trend columns, split across two parquet files.

In [32]:
df = pd.concat([
    pd.read_parquet(DATA_DIR / "polymarket_ml_dataset_with_trends_part1.parquet"),
    pd.read_parquet(DATA_DIR / "polymarket_ml_dataset_with_trends_part2.parquet"),
], ignore_index=True)

df["category"] = df["category"].fillna("other")
df = df.dropna(subset=[TARGET])

train = df[df["split"] == "train"]
test  = df[df["split"] == "test"]

assert len(set(train["market_id"]) & set(test["market_id"])) == 0, "Market leakage detected"

counts         = train.groupby("market_id").size()
sample_weights = train["market_id"].map(counts).rdiv(1).values

y_train    = train[TARGET]
y_test     = test[TARGET]
test_reset = test.reset_index(drop=True)

print(f"Total rows : {len(df):,}  |  Columns: {df.shape[1]}")
print(f"Train      : {len(train):,}  |  {train['market_id'].nunique():,} markets")
print(f"Test       : {len(test):,}   |  {test['market_id'].nunique():,} markets")
print(f"Trend coverage (has_trend_data=1): {df['has_trend_data'].mean():.1%}")
df.head()

Total rows : 4,078,109  |  Columns: 32
Train      : 3,268,545  |  17,073 markets
Test       : 809,564   |  4,243 markets
Trend coverage (has_trend_data=1): 99.1%


,market_id,snapshot_timestamp,days_before_close,pct_lifetime_elapsed,duration_days,price_at_snapshot,price_deviation_from_half,total_volume,log_volume,outcome,...,price_range_14d,price_trend_14d,split,category,question,trend_value,trend_ma4,trend_change_4w,trend_spike,has_trend_data
0,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-01 18:45:42.437000+00:00,59.22,0.1912,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000526,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
1,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-02 06:45:42.437000+00:00,58.72,0.1980,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000354,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
2,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-02 18:45:42.437000+00:00,58.22,0.2049,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000215,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
3,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-03 06:45:42.437000+00:00,57.72,0.2117,73,0.03,0.47,40175.18,10.601,0,...,0.03,0.000479,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
4,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-03 18:45:42.437000+00:00,57.22,0.2185,73,0.03,0.47,40175.18,10.601,0,...,0.03,0.000499,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1


---
## Shared Helpers

In [33]:
def build_pipeline(num_cols, cat_cols):
    return Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                ("scaler",  StandardScaler()),
            ]), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ])),
        ("clf", LogisticRegression(
            class_weight="balanced", max_iter=1000,
            solver="lbfgs", C=1.0, random_state=42,
        )),
    ])

def get_threshold(y_true, y_prob):
    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    return float(thresh[np.argmax(f1)]), float(np.max(f1))

def evaluate(y_true, y_prob, label, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "AUC-ROC" : roc_auc_score(y_true, y_prob),
        "PR-AUC"  : average_precision_score(y_true, y_prob),
        "Log-loss": log_loss(y_true, y_prob),
        "Brier"   : brier_score_loss(y_true, y_prob),
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1"      : f1_score(y_true, y_pred),
    }
    print(f"\n{'─'*50}")
    print(f"  {label}  (threshold={threshold:.3f})")
    print(f"{'─'*50}")
    for name, val in metrics.items():
        print(f"  {name:<10}: {val:.4f}")
    print(f"{'─'*50}")
    return metrics

def per_category(test_df, y_prob, threshold):
    rows = []
    for cat in sorted(test_df["category"].unique()):
        mask = test_df["category"] == cat
        yt   = test_df.loc[mask, TARGET].values
        if len(np.unique(yt)) < 2 or len(yt) < 10:
            continue
        yp = y_prob[mask.values]
        rows.append({
            "category": cat,
            "n"       : int(mask.sum()),
            "AUC"     : roc_auc_score(yt, yp),
            "PR-AUC"  : average_precision_score(yt, yp),
            "F1"      : f1_score(yt, (yp >= threshold).astype(int)),
            "YES%"    : float(yt.mean()),
        })
    return pd.DataFrame(rows).set_index("category").sort_values("AUC", ascending=False)

def market_eval(test_df, y_prob, threshold):
    mdf = (
        test_df.assign(pred_prob=y_prob)
        .groupby("market_id")
        .agg(pred_prob=("pred_prob", "mean"), outcome=(TARGET, "first"))
        .reset_index()
    )
    mp, mt = mdf["pred_prob"].values, mdf["outcome"].values
    mpred  = (mp >= threshold).astype(int)
    print(f"Market-level evaluation ({len(mdf):,} markets)")
    print(f"  AUC-ROC  : {roc_auc_score(mt, mp):.4f}")
    print(f"  PR-AUC   : {average_precision_score(mt, mp):.4f}")
    print(f"  Brier    : {brier_score_loss(mt, mp):.4f}")
    print(f"  Accuracy : {accuracy_score(mt, mpred):.4f}")
    print(f"  F1       : {f1_score(mt, mpred):.4f}")
    return {"AUC-ROC": roc_auc_score(mt,mp), "PR-AUC": average_precision_score(mt,mp),
            "Brier": brier_score_loss(mt,mp), "Accuracy": accuracy_score(mt,mpred), "F1": f1_score(mt,mpred)}

---
# Part 1 — Base Model

Price + engineered features only, no Google Trends.

## 1.1 Train

In [34]:
pipe_base = build_pipeline(
    num_cols=NUMERIC_FEATURES,
    cat_cols=CATEGORICAL_FEATURES,
)
pipe_base.fit(train[FEATURES_BASE], y_train, clf__sample_weight=sample_weights)

y_prob_base        = pipe_base.predict_proba(test[FEATURES_BASE])[:, 1]
thresh_base, f1_base = get_threshold(y_test, y_prob_base)
print(f"Optimal threshold: {thresh_base:.3f}  |  F1: {f1_base:.4f}")

Optimal threshold: 0.754  |  F1: 0.8046


## 1.2 Row-Level Evaluation

In [35]:
metrics_base_row = evaluate(y_test, y_prob_base, "Base Model", thresh_base)


──────────────────────────────────────────────────
  Base Model  (threshold=0.754)
──────────────────────────────────────────────────
  AUC-ROC   : 0.9637
  PR-AUC    : 0.8789
  Log-loss  : 0.2392
  Brier     : 0.0655
  Accuracy  : 0.9369
  F1        : 0.8046
──────────────────────────────────────────────────


## 1.3 Per-Category Breakdown

In [36]:
cat_base = per_category(test_reset, y_prob_base, thresh_base)
cat_base.style.format({"AUC": "{:.4f}", "PR-AUC": "{:.4f}", "F1": "{:.4f}", "YES%": "{:.1%}"})

,n,AUC,PR-AUC,F1,YES%
category,,,,,
politics_global,75292,0.9832,0.9249,0.8378,13.7%
geopolitics,32179,0.9814,0.9205,0.8112,14.0%
crypto,79568,0.9804,0.9521,0.9055,33.0%
finance,53428,0.9740,0.9247,0.8190,22.5%
politics_us,144280,0.9686,0.8938,0.8390,19.1%
science_tech,37344,0.9658,0.9045,0.7960,20.4%
entertainment,84253,0.9612,0.8227,0.7599,13.0%
sports,296593,0.9375,0.7652,0.6893,10.4%
other,6627,0.6675,0.5960,0.5207,14.9%


## 1.4 Market-Level Evaluation

In [37]:
metrics_base_mkt = market_eval(test_reset, y_prob_base, thresh_base)

Market-level evaluation (4,243 markets)
  AUC-ROC  : 0.9566
  PR-AUC   : 0.8550
  Brier    : 0.0770
  Accuracy : 0.9225
  F1       : 0.7645


## 1.5 Feature Importance

In [38]:
clf_step  = pipe_base.named_steps["clf"]
prep_step = pipe_base.named_steps["preprocessor"]
cat_names = list(prep_step.named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES))
all_names = NUMERIC_FEATURES + cat_names

importance_df_base = (
    pd.DataFrame({"feature": all_names, "coefficient": clf_step.coef_[0]})
    .assign(abs_coef=lambda d: d["coefficient"].abs())
    .sort_values("abs_coef", ascending=False)
    .drop(columns="abs_coef")
    .reset_index(drop=True)
)

importance_df_base.head(20).style.bar(
    subset=["coefficient"], align="zero", color=["#d65f5f", "#5fba7d"]
)

,feature,coefficient
0,price_at_snapshot,1.162691
1,price_min_14d,0.642902
2,price_max_14d,0.642354
3,log_volume,0.543873
4,category_other,0.426155
5,category_geopolitics,-0.400519
6,duration_days,-0.358458
7,price_mean_14d,-0.339473
8,category_sports,-0.258177
9,category_crypto,-0.242468


---
# Part 2 — Model with Google Trends

Same as Part 1 plus 5 Google Trends features: `trend_value`, `trend_ma4`, `trend_change_4w`, `trend_spike`, `has_trend_data`.

## 2.1 Train

In [39]:
pipe_trends = build_pipeline(
    num_cols=NUMERIC_FEATURES + TRENDS_FEATURES,
    cat_cols=CATEGORICAL_FEATURES,
)
pipe_trends.fit(train[FEATURES_TRENDS], y_train, clf__sample_weight=sample_weights)

y_prob_trends          = pipe_trends.predict_proba(test[FEATURES_TRENDS])[:, 1]
thresh_trends, f1_trends = get_threshold(y_test, y_prob_trends)
print(f"Optimal threshold: {thresh_trends:.3f}  |  F1: {f1_trends:.4f}")

Optimal threshold: 0.786  |  F1: 0.8058


## 2.2 Row-Level Evaluation

In [40]:
metrics_trends_row = evaluate(y_test, y_prob_trends, "Trends Model", thresh_trends)


──────────────────────────────────────────────────
  Trends Model  (threshold=0.786)
──────────────────────────────────────────────────
  AUC-ROC   : 0.9639
  PR-AUC    : 0.8799
  Log-loss  : 0.2386
  Brier     : 0.0653
  Accuracy  : 0.9385
  F1        : 0.8058
──────────────────────────────────────────────────


## 2.3 Per-Category Breakdown

In [41]:
cat_trends = per_category(test_reset, y_prob_trends, thresh_trends)
cat_trends.style.format({"AUC": "{:.4f}", "PR-AUC": "{:.4f}", "F1": "{:.4f}", "YES%": "{:.1%}"})

,n,AUC,PR-AUC,F1,YES%
category,,,,,
politics_global,75292,0.9827,0.9231,0.8403,13.7%
crypto,79568,0.9819,0.9577,0.9077,33.0%
geopolitics,32179,0.9814,0.9198,0.8116,14.0%
finance,53428,0.9740,0.9244,0.8184,22.5%
politics_us,144280,0.9684,0.8943,0.8433,19.1%
science_tech,37344,0.9657,0.9044,0.7882,20.4%
entertainment,84253,0.9611,0.8222,0.7494,13.0%
sports,296593,0.9375,0.7652,0.6908,10.4%
other,6627,0.6683,0.5963,0.5291,14.9%


## 2.4 Market-Level Evaluation

In [42]:
metrics_trends_mkt = market_eval(test_reset, y_prob_trends, thresh_trends)

Market-level evaluation (4,243 markets)
  AUC-ROC  : 0.9566
  PR-AUC   : 0.8553
  Brier    : 0.0769
  Accuracy : 0.9232
  F1       : 0.7599


## 2.5 Feature Importance

Trend features are highlighted in yellow — their magnitude relative to price features shows how much signal they add.

In [43]:
clf_step  = pipe_trends.named_steps["clf"]
prep_step = pipe_trends.named_steps["preprocessor"]
cat_names = list(prep_step.named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES))
all_names = NUMERIC_FEATURES + TRENDS_FEATURES + cat_names

importance_df = (
    pd.DataFrame({"feature": all_names, "coefficient": clf_step.coef_[0]})
    .assign(
        abs_coef = lambda d: d["coefficient"].abs(),
        is_trend = lambda d: d["feature"].isin(TRENDS_FEATURES),
    )
    .sort_values("abs_coef", ascending=False)
    .drop(columns="abs_coef")
    .reset_index(drop=True)
)

print("Trend feature coefficients:")
print(importance_df[importance_df["is_trend"]].to_string(index=False))
print()

importance_df.head(25).style.bar(
    subset=["coefficient"], align="zero", color=["#d65f5f", "#5fba7d"]
).apply(
    lambda col: ["background-color: #fff3cd" if v else "" for v in importance_df.head(25)["is_trend"]],
    axis=0, subset=["feature", "coefficient"]
)

Trend feature coefficients:
        feature  coefficient  is_trend
    trend_value     0.290175      True
      trend_ma4    -0.268190      True
trend_change_4w    -0.110216      True
    trend_spike     0.071445      True
 has_trend_data    -0.053013      True



,feature,coefficient,is_trend
0,price_at_snapshot,1.144267,False
1,price_min_14d,0.649924,False
2,price_max_14d,0.645871,False
3,log_volume,0.545796,False
4,duration_days,-0.370908,False
5,category_geopolitics,-0.337629,False
6,price_mean_14d,-0.319417,False
7,trend_value,0.290175,True
8,trend_ma4,-0.268190,True
9,category_crypto,-0.236990,False


---
# Part 3 — Comparison

Head-to-head: Base vs Trends across row-level metrics, market-level metrics, and per-category AUC.

## 3.1 Row-Level

In [44]:
row_comparison = pd.DataFrame({
    "Base":   metrics_base_row,
    "Trends": metrics_trends_row,
})
row_comparison["Delta"] = row_comparison["Trends"] - row_comparison["Base"]
row_comparison.style.format("{:.4f}").bar(
    subset=["Delta"], align="zero", color=["#d65f5f", "#5fba7d"]
)

,Base,Trends,Delta
AUC-ROC,0.9637,0.9639,0.0002
PR-AUC,0.8789,0.8799,0.0010
Log-loss,0.2392,0.2386,-0.0006
Brier,0.0655,0.0653,-0.0002
Accuracy,0.9369,0.9385,0.0017
F1,0.8046,0.8058,0.0012


## 3.2 Market-Level

In [45]:
mkt_comparison = pd.DataFrame({
    "Base":   metrics_base_mkt,
    "Trends": metrics_trends_mkt,
})
mkt_comparison["Delta"] = mkt_comparison["Trends"] - mkt_comparison["Base"]
mkt_comparison.style.format("{:.4f}").bar(
    subset=["Delta"], align="zero", color=["#d65f5f", "#5fba7d"]
)

,Base,Trends,Delta
AUC-ROC,0.9566,0.9566,0.0001
PR-AUC,0.8550,0.8553,0.0003
Brier,0.0770,0.0769,-0.0001
Accuracy,0.9225,0.9232,0.0007
F1,0.7645,0.7599,-0.0046


## 3.3 Per-Category AUC Delta

In [46]:
cat_comparison = cat_base[["n", "AUC", "YES%"]].rename(columns={"AUC": "AUC (base)"})
cat_comparison["AUC (trends)"] = cat_trends["AUC"]
cat_comparison["ΔAUC"] = cat_comparison["AUC (trends)"] - cat_comparison["AUC (base)"]
cat_comparison = cat_comparison.sort_values("ΔAUC", ascending=False)
cat_comparison.style.format({
    "AUC (base)": "{:.4f}", "AUC (trends)": "{:.4f}",
    "ΔAUC": "{:+.4f}", "YES%": "{:.1%}",
}).bar(subset=["ΔAUC"], align="zero", color=["#d65f5f", "#5fba7d"])

,n,AUC (base),YES%,AUC (trends),ΔAUC
category,,,,,
crypto,79568,0.9804,33.0%,0.9819,+0.0015
other,6627,0.6675,14.9%,0.6683,+0.0008
finance,53428,0.9740,22.5%,0.9740,+0.0000
sports,296593,0.9375,10.4%,0.9375,-0.0000
geopolitics,32179,0.9814,14.0%,0.9814,-0.0000
science_tech,37344,0.9658,20.4%,0.9657,-0.0001
entertainment,84253,0.9612,13.0%,0.9611,-0.0001
politics_us,144280,0.9686,19.1%,0.9684,-0.0002
politics_global,75292,0.9832,13.7%,0.9827,-0.0005


---
## Save Artifacts

In [47]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump({"pipeline": pipe_base},   ARTIFACTS_DIR / "model_base.joblib")
joblib.dump({"pipeline": pipe_trends}, ARTIFACTS_DIR / "model_trends.joblib")
print(f"Models saved → {ARTIFACTS_DIR}")

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
pred_df = test[["market_id", "category", TARGET]].copy().reset_index(drop=True)
pred_df["pred_prob_base"]    = y_prob_base
pred_df["pred_prob_trends"]  = y_prob_trends
pred_df["pred_label_base"]   = (y_prob_base   >= thresh_base).astype(int)
pred_df["pred_label_trends"] = (y_prob_trends >= thresh_trends).astype(int)
preds_path = PREDICTIONS_DIR / "predictions.csv"
pred_df.to_csv(preds_path, index=False)
print(f"Predictions saved → {preds_path}")
pred_df.head()

Models saved → artifacts
Predictions saved → predictions/predictions.csv


,market_id,category,outcome,pred_prob_base,pred_prob_trends,pred_label_base,pred_label_trends
0,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.730942,0.719568,0,0
1,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.798754,0.788649,1,1
2,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.793634,0.784776,1,0
3,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.804380,0.794728,1,1
4,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,crypto,0,0.893868,0.888577,1,1
